# Trivima — SD + LoRA + CLIP (resume + fast)

**Features:**
- CLIP ViT-L image encoder for strong scene conditioning
- Classifier-free guidance training (10% drop)
- **Auto-resume** from Drive checkpoint (survives runtime restart / disconnect)
- **Optimized data pipeline** (persistent workers, prefetch, more workers)
- Saves to Drive every epoch

**Run all:** Runtime → Run all. Click through Drive auth popup when it appears.

## 1. Install + mount Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q diffusers transformers peft accelerate datasets huggingface_hub pyarrow

# Mount Drive — REQUIRED for resume capability
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/trivima_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Checkpoint dir: {DRIVE_DIR}')
!ls -lh {DRIVE_DIR}/ 2>/dev/null || echo 'no existing checkpoints'

## 2. Load SD 1.5 + LoRA + CLIP

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler
from peft import LoraConfig, get_peft_model
from transformers import CLIPVisionModel

device = 'cuda'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

vae = AutoencoderKL.from_pretrained('stabilityai/sd-vae-ft-mse').to(device)
vae.requires_grad_(False); vae.eval()

clip_vision = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14').to(device)
clip_vision.requires_grad_(False); clip_vision.eval()
CLIP_DIM = 1024

unet = UNet2DConditionModel.from_pretrained(
    'stable-diffusion-v1-5/stable-diffusion-v1-5', subfolder='unet'
).to(device)

# Enable PyTorch 2 SDPA flash attention (free speedup, no install)
try:
    from diffusers.models.attention_processor import AttnProcessor2_0
    unet.set_attn_processor(AttnProcessor2_0())
    print('SDPA flash attention enabled')
except Exception as e:
    print(f'SDPA setup warning: {e}')

lora_config = LoraConfig(
    r=16, lora_alpha=16, lora_dropout=0.1,
    target_modules=['to_q', 'to_v', 'to_k', 'to_out.0'],
)
unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

clip_proj = nn.Sequential(
    nn.LayerNorm(CLIP_DIM),
    nn.Linear(CLIP_DIM, 768),
).to(device)

scheduler = DDPMScheduler(num_train_timesteps=1000)
scheduler.alpha_cumprod = scheduler.alphas_cumprod.to(device)
print('Models ready')

## 3. Download dataset

In [ ]:
from huggingface_hub import snapshot_download

os.makedirs('/content/data/re10k_hf', exist_ok=True)
snapshot_download(
    repo_id='Wouter01/re10k_pixelsplat_hard',
    repo_type='dataset',
    local_dir='/content/data/re10k_hf',
    allow_patterns=['data/train-0000[0-3]-of-00036.parquet'],
    max_workers=4,
)
!ls -lh /content/data/re10k_hf/data/

## 4. Dataset + fast DataLoader

In [ ]:
import io, glob
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pyarrow.parquet as pq
import torchvision.transforms as T

CLIP_MEAN = [0.48145466, 0.4578275, 0.40821073]
CLIP_STD  = [0.26862954, 0.26130258, 0.27577711]

class RE10KParquet(Dataset):
    def __init__(self, parquet_glob, sd_size=256, max_pairs=None):
        self.files = sorted(glob.glob(parquet_glob))
        # Pre-load all parquet tables ONCE in main process so workers share via fork
        print('Loading parquet tables into RAM...')
        self.tables = [pq.read_table(p) for p in self.files]
        self.index = []
        for fi, t in enumerate(self.tables):
            n = t.num_rows
            for r in range(n):
                self.index.append((fi, r))
                if max_pairs and len(self.index) >= max_pairs: break
            if max_pairs and len(self.index) >= max_pairs: break
        self.sd_tf = T.Compose([
            T.Resize((sd_size, sd_size), T.InterpolationMode.LANCZOS),
            T.ToTensor(),
            T.Normalize([0.5]*3, [0.5]*3),
        ])
        self.clip_tf = T.Compose([
            T.Resize((224, 224), T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize(CLIP_MEAN, CLIP_STD),
        ])
        print(f'Dataset: {len(self.index)} pairs from {len(self.files)} shards')

    def __len__(self):
        return len(self.index)

    def __getitem__(self, i):
        fi, ri = self.index[i]
        t = self.tables[fi]
        cond = t.column('conditioning_image')[ri].as_py()['bytes']
        gt   = t.column('ground_truth_image')[ri].as_py()['bytes']
        cond_img = Image.open(io.BytesIO(cond)).convert('RGB')
        gt_img   = Image.open(io.BytesIO(gt)).convert('RGB')
        return {
            'input_sd':   self.sd_tf(cond_img),
            'input_clip': self.clip_tf(cond_img),
            'target':     self.sd_tf(gt_img),
        }

dataset = RE10KParquet('/content/data/re10k_hf/data/train-*.parquet', sd_size=256, max_pairs=10000)
loader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=8,
    drop_last=True,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
)
print(f'Batches/epoch: {len(loader)}')

## 5. Train (auto-resume from Drive, save every epoch)

In [ ]:
import time, numpy as np

EPOCHS = 20
LR = 1e-4
COND_DROPOUT = 0.1
CKPT_PATH = f'{DRIVE_DIR}/best_clip.pt'
RESUME_PATH = f'{DRIVE_DIR}/last_clip.pt'  # full state for resume

trainable = list(unet.parameters()) + list(clip_proj.parameters())
optimizer = torch.optim.AdamW([p for p in trainable if p.requires_grad], lr=LR, weight_decay=0.01)

best_loss = float('inf')
start_epoch = 0

# === RESUME ===
if os.path.exists(RESUME_PATH):
    print(f'Found {RESUME_PATH} — resuming...')
    ck = torch.load(RESUME_PATH, map_location=device)
    unet.load_state_dict(ck['unet'], strict=False)
    clip_proj.load_state_dict(ck['clip_proj'])
    optimizer.load_state_dict(ck['optimizer'])
    start_epoch = ck['epoch'] + 1
    best_loss = ck.get('best_loss', float('inf'))
    print(f'Resumed from epoch {start_epoch}, best_loss={best_loss:.4f}')
else:
    print('No checkpoint — starting from scratch')

def save_best(loss, epoch):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'clip_proj': clip_proj.state_dict(),
        'loss': loss, 'epoch': epoch,
    }, CKPT_PATH)

def save_resume(loss, epoch):
    torch.save({
        'unet': {k: v for k, v in unet.state_dict().items() if 'lora' in k.lower()},
        'clip_proj': clip_proj.state_dict(),
        'optimizer': optimizer.state_dict(),
        'epoch': epoch, 'best_loss': best_loss,
    }, RESUME_PATH)

for epoch in range(start_epoch, EPOCHS):
    t0 = time.time()
    losses = []
    unet.train(); clip_proj.train()

    for bi, batch in enumerate(loader):
        inp_sd   = batch['input_sd'].to(device, non_blocking=True)
        inp_clip = batch['input_clip'].to(device, non_blocking=True)
        tgt      = batch['target'].to(device, non_blocking=True)
        B = inp_sd.shape[0]

        with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
            tgt_lat = vae.encode(tgt).latent_dist.sample() * 0.18215
            clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state

        with torch.autocast('cuda', dtype=torch.bfloat16):
            enc_h = clip_proj(clip_feats)  # (B, 257, 768)

            if COND_DROPOUT > 0:
                drop_mask = (torch.rand(B, device=device) < COND_DROPOUT).view(B, 1, 1)
                enc_h = torch.where(drop_mask, torch.zeros_like(enc_h), enc_h)

            t = torch.randint(0, 1000, (B,), device=device).long()
            noise = torch.randn_like(tgt_lat)
            noisy = scheduler.add_noise(tgt_lat, noise, t)

            pred = unet(noisy, t, encoder_hidden_states=enc_h).sample
            loss = F.mse_loss(pred.float(), noise.float())

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        optimizer.step()

        losses.append(loss.item())
        if (bi+1) % 20 == 0:
            print(f'  [{epoch+1}/{EPOCHS}] {bi+1}/{len(loader)} loss={np.mean(losses[-20:]):.4f}')

    avg = float(np.mean(losses))
    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS}: loss={avg:.4f} ({dt:.0f}s)')

    # Save resume checkpoint EVERY epoch (so disconnect → resume works)
    save_resume(avg, epoch)

    if avg < best_loss:
        best_loss = avg
        save_best(best_loss, epoch)
        print(f'  -> new best: {best_loss:.4f}')

print(f'\nDone! Best loss: {best_loss:.4f}')

## 6. Generate with classifier-free guidance

In [ ]:
import matplotlib.pyplot as plt

ckpt = torch.load(CKPT_PATH, map_location=device)
unet.load_state_dict(ckpt['unet'], strict=False)
clip_proj.load_state_dict(ckpt['clip_proj'])
unet.eval(); clip_proj.eval()
print(f"Loaded, best loss: {ckpt['loss']:.4f}")

batch = next(iter(loader))
inp_sd   = batch['input_sd'][:1].to(device)
inp_clip = batch['input_clip'][:1].to(device)
gt       = batch['target'][:1].to(device)

GUIDANCE = 5.0
STRENGTH = 0.6

with torch.no_grad():
    inp_lat = vae.encode(inp_sd).latent_dist.sample() * 0.18215
    clip_feats = clip_vision(pixel_values=inp_clip).last_hidden_state
    enc_cond = clip_proj(clip_feats)
    enc_uncond = torch.zeros_like(enc_cond)

    n_steps = 50
    start_step = int(n_steps * STRENGTH)
    timesteps = torch.linspace(999, 0, n_steps, device=device).long()
    start_t = timesteps[n_steps - start_step]

    noise = torch.randn_like(inp_lat)
    latent = scheduler.add_noise(inp_lat, noise, start_t.unsqueeze(0))

    for idx in range(n_steps - start_step, n_steps):
        i = int(timesteps[idx])
        i_next = int(timesteps[idx + 1]) if idx + 1 < n_steps else 0
        t = torch.full((1,), i, device=device, dtype=torch.long)
        np_uncond = unet(latent, t, encoder_hidden_states=enc_uncond).sample
        np_cond   = unet(latent, t, encoder_hidden_states=enc_cond).sample
        noise_pred = np_uncond + GUIDANCE * (np_cond - np_uncond)

        a_t = scheduler.alpha_cumprod[i]
        a_next = scheduler.alpha_cumprod[i_next] if i_next > 0 else torch.tensor(1.0, device=device)
        pred_x0 = (latent - torch.sqrt(1 - a_t) * noise_pred) / torch.sqrt(a_t)
        pred_x0 = pred_x0.clamp(-4, 4)
        latent = torch.sqrt(a_next) * pred_x0 + torch.sqrt(1 - a_next) * noise_pred

    gen = vae.decode(latent / 0.18215).sample
    gen = (gen.clamp(-1, 1) + 1) / 2

def to_np(x):
    return x[0].cpu().permute(1, 2, 0).float().numpy().clip(0, 1)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(to_np((inp_sd + 1) / 2)); axes[0].set_title('Input Photo')
axes[1].imshow(to_np(gen)); axes[1].set_title(f'Generated (CFG={GUIDANCE} str={STRENGTH})')
axes[2].imshow(to_np((gt + 1) / 2)); axes[2].set_title('Ground Truth')
for ax in axes: ax.axis('off')
plt.tight_layout()
plt.savefig('/content/sd_clip_result.png', dpi=150)
plt.show()

mse = ((gen - (gt + 1) / 2) ** 2).mean().item()
psnr = -10 * np.log10(mse) if mse > 0 else float('inf')
print(f'PSNR: {psnr:.2f} dB')